In [5]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split

In [6]:
%pip install mlflow dagshub

In [7]:
import dagshub
dagshub.init(repo_owner='xlrboi', repo_name='delievery_time_prediction', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f03cdbbe-8b45-4826-9ffa-a8567dfdaa08&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e77d378127394f3de069a08940d2c1e60218f4f914657eaebca0201e12cae952




Accessing as xlrboi

Initialized MLflow to track repo "xlrboi/delievery_time_prediction"

Repository xlrboi/delievery_time_prediction initialized!

In [8]:
import mlflow

In [9]:
mlflow.set_tracking_uri("https://dagshub.com/xlrboi/delievery_time_prediction.mlflow")

In [10]:
mlflow.set_experiment("4 - Stacking Regressor")

<Experiment: artifact_location='mlflow-artifacts:/cf1c53c9044c4f4abb19cefb0ea6357d', creation_time=1784404419663, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1784404419663, lifecycle_stage='active', name='4 - Stacking Regressor', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [11]:
from sklearn import set_config

set_config(transform_output="pandas")

In [12]:
df = pd.read_csv('cleaned_data.csv')
df

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37690,35.0,4.2,windy,jam,2,drinks,motorcycle,1.0,no,metropolitian,33,0,10.0,night,16.600272,very_long
37691,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
37692,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
37693,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [13]:
temp_df = df.copy().dropna()

In [14]:
X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

X

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37690,35.0,4.2,windy,jam,2,drinks,motorcycle,1.0,no,metropolitian,0,10.0,night,16.600272,very_long
37691,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,0,10.0,morning,1.489846,short
37692,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,0,15.0,night,4.657195,short
37693,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,0,5.0,afternoon,6.232393,medium


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [16]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (30156, 15)
The shape of test data is (7539, 15)


In [17]:
pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [18]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [19]:
nominal_cat_cols

['weather',
 'type_of_order',
 'type_of_vehicle',
 'festival',
 'city_type',
 'is_weekend',
 'order_time_of_day']

In [20]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [21]:
# unique categories the ordinal columns

for col in ordinal_cat_cols:
    print(col,X_train[col].unique())

traffic ['jam' 'medium' 'high' 'low']
distance_type ['medium' 'short' 'long' 'very_long']


In [22]:
# build a preprocessor

preprocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",
                                     sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                      encoded_missing_value=-999,
                                      handle_unknown="use_encoded_value",
                                      unknown_value=-1), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)


preprocessor

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['age', 'ratings', 'pickup_time_minutes',
                                  'distance']),
                                ('nominal_encode',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['weather', 'type_of_order', 'type_of_vehicle',
                                  'festival', 'city_type', 'is_weekend',
                                  'order_time_of_day']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['low', 'medium',
                                                             'high', 'jam'],
                                                            ['short', 'medium',
                                                             'long',
                                                             'very_long']],
                                                encoded_missing_value=-999,
                                                handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['traffic', 'distance_type'])],
                  verbose_feature_names_out=False)

In [23]:
# build the pipeline

processing_pipeline = Pipeline(steps=[
                                # ("simple_imputer",simple_imputer),
                                ("preprocess",preprocessor)
                                # ("knn_imputer",knn_imputer)
                            ])

processing_pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                                   remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  ['age', 'ratings',
                                                   'pickup_time_minutes',
                                                   'distance']),
                                                 ('nominal_encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['weather', 'type_of_order',
                                                   'type_of_vehicle',
                                                   'festival', 'city_type',
                                                   'is_weekend',
                                                   'order_time_of_day']),
                                                 ('ordinal_encode',
                                                  OrdinalEncoder(categories=[['low',
                                                                              'medium',
                                                                              'high',
                                                                              'jam'],
                                                                             ['short',
                                                                              'medium',
                                                                              'long',
                                                                              'very_long']],
                                                                 encoded_missing_value=-999,
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['traffic',
                                                   'distance_type'])],
                                   verbose_feature_names_out=False))])

In [24]:
# do data preprocessing

X_train_trans = processing_pipeline.fit_transform(X_train)

X_test_trans = processing_pipeline.transform(X_test)

In [25]:
X_train_trans

,age,ratings,pickup_time_minutes,distance,weather_fog,weather_sandstorms,weather_stormy,weather_sunny,weather_windy,type_of_order_drinks,...,city_type_semi-urban,city_type_urban,is_weekend_1,order_time_of_day_evening,order_time_of_day_morning,order_time_of_day_night,traffic,distance_type,vehicle_condition,multiple_deliveries
7204,0.473684,0.56,1.0,0.404165,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,3.0,1.0,0,2.0
20974,1.000000,0.76,0.0,0.154044,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0,1.0
28236,0.473684,0.80,0.5,0.002461,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1,0.0
21635,1.000000,0.92,1.0,0.460411,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0,1.0
30780,0.526316,0.76,0.5,0.243676,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16850,0.578947,0.92,0.5,0.451895,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,3.0,2.0,0,0.0
6265,0.052632,1.00,1.0,0.612270,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,2.0,1,1.0
11284,0.526316,0.92,0.0,0.322877,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1,0.0
860,0.947368,0.96,0.5,0.004486,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0,1.0


In [26]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 12.7 MB/s eta 0:00:00


In [27]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
from sklearn.metrics import mean_absolute_error

In [28]:
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import StackingRegressor

In [29]:
# build the best models

best_xgb_params = {'n_estimators': 856,
 'learning_rate': 0.011804738897525,
 'max_depth': 12,
 'min_child_weight': 6,
 'gamma': 0.6035875191188472,
 'subsample': 0.9028989534142778,
 'colsample_bytree': 0.9791480707107688,
 'reg_alpha': 0.00020972186331087944,
 'reg_lambda': 0.0033654731421510226}

best_lgbm_params = {'n_estimators': 609,
 'learning_rate': 0.011840062109564906,
 'max_depth': 15,
 'num_leaves': 107,
 'subsample': 0.8342949072227168,
 'colsample_bytree': 0.9339475480969448,
 'min_child_weight': 5.802155537397966,
 'reg_alpha': 0.000172409806797029,
 'reg_lambda': 0.2470475418456122}

best_xgb = XGBRegressor(**best_xgb_params)
best_lgbm = LGBMRegressor(**best_lgbm_params)

In [31]:
from sklearn.linear_model import Ridge,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_validate
import optuna
import mlflow

def objective(trial):

    with mlflow.start_run(nested=True):

        model_name=trial.suggest_categorical(
            "meta_model",
            ["Ridge","ElasticNet","DT"]
        )

        if model_name=="Ridge":

            meta=Ridge(
                alpha=trial.suggest_float("ridge_alpha",0.01,20,log=True),
                random_state=42
            )

        elif model_name=="ElasticNet":

            meta=ElasticNet(
                alpha=trial.suggest_float("enet_alpha",0.001,5,log=True),
                l1_ratio=trial.suggest_float("l1_ratio",0.2,0.8),
                max_iter=3000,
                random_state=42
            )

        else:

            meta=DecisionTreeRegressor(
                max_depth=trial.suggest_int("max_depth",3,10),
                min_samples_leaf=trial.suggest_int("min_samples_leaf",1,5),
                random_state=42
            )

        stack=StackingRegressor(
            estimators=[
                ("xgb",best_xgb),
                ("lgbm",best_lgbm)
            ],
            final_estimator=meta,
            cv=3,
            n_jobs=-1
        )

        model=TransformedTargetRegressor(
            regressor=stack,
            transformer=pt
        )

        scores=cross_validate(
            estimator=model,
            X=X_train_trans,
            y=y_train,
            cv=3,
            scoring="neg_mean_absolute_error",
            return_train_score=True,
            n_jobs=-1
        )

        cv_mae=-scores["test_score"].mean()
        train_mae=-scores["train_score"].mean()

        mlflow.log_params(trial.params)
        mlflow.log_metric("cv_mae",cv_mae)
        mlflow.log_metric("train_mae",train_mae)
        mlflow.log_metric("cv_std",scores["test_score"].std())
        mlflow.log_metric("fit_time",scores["fit_time"].mean())

        return cv_mae


study=optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=2
    )
)

with mlflow.start_run(run_name="Stacking_Optuna"):

    study.optimize(
        objective,
        n_trials=25,
        n_jobs=1,
        show_progress_bar=True
    )

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_cv_mae",study.best_value)

    params=study.best_params.copy()
    model_name=params.pop("meta_model")

    if model_name=="Ridge":

        meta=Ridge(
            alpha=params["ridge_alpha"],
            random_state=42
        )

    elif model_name=="ElasticNet":

        meta=ElasticNet(
            alpha=params["enet_alpha"],
            l1_ratio=params["l1_ratio"],
            max_iter=3000,
            random_state=42
        )

    else:

        meta=DecisionTreeRegressor(
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42
        )

    stack=StackingRegressor(
        estimators=[
            ("xgb",best_xgb),
            ("lgbm",best_lgbm)
        ],
        final_estimator=meta,
        cv=3,
        n_jobs=-1
    )

    final_model=TransformedTargetRegressor(
        regressor=stack,
        transformer=pt
    )

    final_model.fit(X_train_trans,y_train)

    y_pred_train=final_model.predict(X_train_trans)
    y_pred_test=final_model.predict(X_test_trans)

    train_mae=mean_absolute_error(y_train,y_pred_train)
    test_mae=mean_absolute_error(y_test,y_pred_test)

    train_r2=r2_score(y_train,y_pred_train)
    test_r2=r2_score(y_test,y_pred_test)

    mlflow.log_metric("train_mae",train_mae)
    mlflow.log_metric("test_mae",test_mae)
    mlflow.log_metric("train_r2",train_r2)
    mlflow.log_metric("test_r2",test_r2)


print("Best Parameters:",study.best_params)
print("Best CV MAE:",study.best_value)
print("Train MAE:",train_mae)
print("Test MAE:",test_mae)
print("Train R2:",train_r2)
print("Test R2:",test_r2)

[I 2026-07-19 06:28:44,144] A new study created in memory with name: no-name-211b5d06-fa75-4138-b781-1e8ba4895862


  0%|          | 0/25 [00:00<?, ?it/s]

🏃 View run sedate-snipe-588 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/4/runs/f86e5d5b082740529e44c2f4fe9df96a
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/4
[I 2026-07-19 06:29:47,959] Trial 0 finished with value: 3.118913440661231 and parameters: {'meta_model': 'ElasticNet', 'enet_alpha': 0.16383993835282307, 'l1_ratio': 0.29361118426546196}. Best is trial 0 with value: 3.118913440661231.
🏃 View run victorious-kite-258 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/4/runs/fc1df1ae71804cc4905900629bc0993a
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/4
[I 2026-07-19 06:30:43,648] Trial 1 finished with value: 3.061805758048999 and parameters: {'meta_model': 'DT', 'max_depth': 7, 'min_samples_leaf': 4}. Best is trial 1 with value: 3.061805758048999.
🏃 View run debonair-smelt-873 at: https://dagshub.com/xlrboi/delievery_tim

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ElasticNet was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ElasticNet was fitted without feature names
  warnings.warn(


🏃 View run Stacking_Optuna at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/4/runs/6791f4cd5acf499aa4aef12d863b42ec
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/4
Best Parameters: {'meta_model': 'ElasticNet', 'enet_alpha': 0.01747401579628145, 'l1_ratio': 0.5351148384675742}
Best CV MAE: 3.02724370554207
Train MAE: 2.7245885812355994
Test MAE: 2.971048710651963
Train R2: 0.8717414006273618
Test R2: 0.8415990978109097


In [34]:
mlflow.sklearn.log_model(
        final_model,
        artifact_path="stacking_model",skops_trusted_types=[
        'xgboost.core.Booster',
        'xgboost.sklearn.XGBRegressor',
        'collections.OrderedDict', 'lightgbm.basic.Booster', 'lightgbm.sklearn.LGBMRegressor', 'sklearn.utils._bunch.Bunch'
    ],
    )

2026/07/19 06:53:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [35]:
study.best_params

{'meta_model': 'ElasticNet',
 'enet_alpha': 0.01747401579628145,
 'l1_ratio': 0.5351148384675742}

In [36]:
study.best_value

3.02724370554207

In [37]:
optuna.visualization.plot_optimization_history(study)

In [38]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_enet_alpha,params_l1_ratio,params_max_depth,params_meta_model,params_min_samples_leaf,params_ridge_alpha,state
0,0,3.118913,2026-07-19 06:28:44.615401,2026-07-19 06:29:47.958946,0 days 00:01:03.343545,0.163840,0.293611,NaN,ElasticNet,NaN,NaN,COMPLETE
1,1,3.061806,2026-07-19 06:29:47.962624,2026-07-19 06:30:43.648310,0 days 00:00:55.685686,NaN,NaN,7.0,DT,4.0,NaN,COMPLETE
2,2,3.028126,2026-07-19 06:30:43.651161,2026-07-19 06:31:38.332270,0 days 00:00:54.681109,0.006101,0.309095,NaN,ElasticNet,NaN,NaN,COMPLETE
3,3,3.048095,2026-07-19 06:31:38.335299,2026-07-19 06:32:32.972452,0 days 00:00:54.637153,NaN,NaN,6.0,DT,2.0,NaN,COMPLETE
4,4,3.029528,2026-07-19 06:32:32.974693,2026-07-19 06:33:26.411199,0 days 00:00:53.436506,NaN,NaN,NaN,Ridge,NaN,0.161946,COMPLETE
5,5,3.056611,2026-07-19 06:33:26.413195,2026-07-19 06:34:21.116958,0 days 00:00:54.703763,0.079825,0.555449,NaN,ElasticNet,NaN,NaN,COMPLETE
6,6,3.028974,2026-07-19 06:34:21.119359,2026-07-19 06:35:14.853285,0 days 00:00:53.733926,0.001740,0.769331,NaN,ElasticNet,NaN,NaN,COMPLETE
7,7,3.029531,2026-07-19 06:35:14.855388,2026-07-19 06:36:08.610069,0 days 00:00:53.754681,NaN,NaN,NaN,Ridge,NaN,0.021010,COMPLETE
8,8,3.029522,2026-07-19 06:36:08.613568,2026-07-19 06:37:02.822026,0 days 00:00:54.208458,NaN,NaN,NaN,Ridge,NaN,0.431116,COMPLETE
9,9,3.345413,2026-07-19 06:37:02.824256,2026-07-19 06:37:56.884412,0 days 00:00:54.060156,0.282260,0.387027,NaN,ElasticNet,NaN,NaN,COMPLETE


In [39]:
optuna.visualization.plot_optimization_history(study)